# Suffix Automaton & Suffix Tree

The **suffix structures** notebook organized a text's suffixes into a *sorted array* (with an LCP array) — light, simple, and enough for most substring queries. This notebook builds the two **heaviest** string structures, the ones that pack *every* substring into a single deterministic machine:

- the **suffix automaton (SAM)** — the **minimal DFA that accepts exactly the substrings of `s`**, built **online in O(n)**, and
- the **suffix tree** — the compressed trie of all suffixes, the SAM's mirror image (the SAM of the *reversed* string is the suffix tree's suffix-link tree).

The suffix automaton is the star: in O(n) states it answers "is this a substring?", "how many distinct substrings?", and "what's the longest common substring of two texts?" — each one *proven* below against a brute force over every substring of small strings (`"abcbc"`, `"banana"`). The suffix tree we build naively, just to *see* the structure and the SAM↔tree duality; production builds use **Ukkonen's** O(n) algorithm, which we name but don't grind out.

**Contents**

1. **What a suffix automaton is** — the minimal DFA for all substrings
2. **Suffix automaton** — online O(n) construction (link, len, clone-on-split), from scratch
3. **Applications** — count distinct substrings; longest common substring of two strings
4. **Suffix tree** — the SAM↔tree relationship, Ukkonen named, a small illustration
5. **When to use what** — suffix array vs automaton vs tree
6. **Complexity cheat-sheet**

In [1]:
import random

rng = random.Random(20240602)              # seeded -> every 'random' check below is reproducible

def all_substrings(s):                     # ground truth: the SET of every substring of s
    return {s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)}

print("all_substrings('abc') ->", sorted(all_substrings('abc')))
print("count for 'banana'    ->", len(all_substrings('banana')))   # 15 distinct substrings

all_substrings('abc') -> ['a', 'ab', 'abc', 'b', 'bc', 'c']
count for 'banana'    -> 15


## 1. What a suffix automaton is

Think of every substring of `s` as a *string the machine should accept*. A **suffix automaton (SAM)** is the **smallest deterministic finite automaton (DFA)** that accepts **exactly** those strings — feed it a word, follow one transition per character from the start state, and you arrive somewhere iff the word is a substring of `s`. No backtracking, no separators: it's a pure character-driven walk, just like the **trie** of the **tries** notebook, but folded down to the minimum number of states.

Two facts make it remarkable:

- **It's tiny.** For a string of length `n` the SAM has **at most `2n - 1` states** and `3n - 4` transitions (for `n >= 3`) — *linear*, even though `s` can have up to `~n²/2` distinct substrings. Substrings that always occur in the same set of positions (the same *endpos* set) share a state.
- **It's built online in O(n).** You add one character at a time, and the automaton for the longer string is patched from the shorter one — no rebuild.

The organizing concept is **endpos**: for a substring `w`, `endpos(w)` is the set of positions in `s` where `w` ends. Two substrings with identical endpos sets are *indistinguishable* by what can follow them, so the SAM merges them into one state. Each state therefore represents a **range of substring lengths** sharing an endpos class, bounded by `len` (longest) and `len[link]` + 1 (shortest) — the fact we'll exploit to count distinct substrings.

## 2. Suffix automaton — online O(n) construction

Each state carries three things:

| Field | Meaning |
|---|---|
| `nxt[v]` | a `dict` of `char -> state`: the DFA transitions |
| `len[v]` | length of the **longest** substring landing in state `v` |
| `link[v]` | the **suffix link** — to the state of the longest proper suffix that lives in a *different* endpos class |

The suffix links form a tree rooted at the initial state (`0`); they're the same "fall back to a shorter suffix" idea as **KMP**'s prefix function and **Aho-Corasick**'s fail links (see the **suffix structures** notebook), here threaded through the automaton's states.

**`extend(ch)` — add one character.** Create a fresh state `cur` for the whole new string. Then walk suffix links from the previous end state `last`, adding a `ch`-transition to `cur` wherever none exists. Three cases close it out:

1. **Hit the root** with no existing `ch`-edge → `link[cur] = 0`.
2. **Found a state `p` with a `ch`-edge to `q`, and `len[p] + 1 == len[q]`** → `q` is already exactly the right suffix; `link[cur] = q`.
3. **Otherwise `q` is "too long"** (it represents longer strings than the suffix we need) → **clone** `q`: a copy `clone` with the same transitions and link but `len = len[p] + 1`, redirect the `ch`-edges that pointed to `q` (from `p` up its suffix-link chain) to `clone`, and set `link[q] = link[cur] = clone`. This **split** is the one subtle move; it's what keeps the automaton minimal.

The whole loop is amortized O(n): the clone case happens rarely enough that the total work across all `extend` calls is linear.

In [2]:
class SuffixAutomaton:
    def __init__(self):
        self.nxt = [{}]        # nxt[v][ch] -> state   (the DFA transitions)
        self.link = [-1]       # link[v]    -> suffix-link target (-1 only for the root)
        self.length = [0]      # length[v]  -> longest substring ending in state v
        self.last = 0          # state representing the whole current string

    def extend(self, ch):
        cur = self._new(self.length[self.last] + 1)
        p = self.last
        while p != -1 and ch not in self.nxt[p]:
            self.nxt[p][ch] = cur                 # add ch-edge along the suffix-link chain
            p = self.link[p]
        if p == -1:                               # case 1: fell off the top
            self.link[cur] = 0
        else:
            q = self.nxt[p][ch]
            if self.length[p] + 1 == self.length[q]:
                self.link[cur] = q                # case 2: q is exactly the suffix we need
            else:                                 # case 3: split q by cloning it
                clone = self._new(self.length[p] + 1)
                self.nxt[clone] = dict(self.nxt[q])
                self.link[clone] = self.link[q]
                while p != -1 and self.nxt[p].get(ch) == q:
                    self.nxt[p][ch] = clone       # redirect edges p..->q onto the clone
                    p = self.link[p]
                self.link[q] = self.link[cur] = clone
        self.last = cur

    def _new(self, length):
        self.nxt.append({}); self.link.append(-1); self.length.append(length)
        return len(self.nxt) - 1

    def accepts(self, w):                         # walk the DFA: substring iff the path exists
        v = 0
        for ch in w:
            v = self.nxt[v].get(ch, -1)
            if v == -1:
                return False
        return True

def build_sam(s):
    sam = SuffixAutomaton()
    for ch in s:
        sam.extend(ch)
    return sam

sam = build_sam('abcbc')
print('states for "abcbc":', len(sam.length), '(<= 2n-1 =', 2 * 5 - 1, ')')
print('accepts "bcb" :', sam.accepts('bcb'))     # substring  -> True
print('accepts "cba" :', sam.accepts('cba'))     # not present -> False
print('accepts ""    :', sam.accepts(''))        # empty string is a substring -> True

states for "abcbc": 8 (<= 2n-1 = 9 )
accepts "bcb" : True
accepts "cba" : False
accepts ""    : True


### Proof: the SAM accepts *exactly* the substrings of `s`

The defining claim is a set equality: `{w : sam.accepts(w)} == all_substrings(s) ∪ {""}`. We verify both directions on thousands of small random strings — **every** substring must be accepted, and a sample of non-substrings must be rejected — and we check the `2n - 1` state bound holds.

In [3]:
# PROOF: accepts(w) is True iff w is a substring of s (empty string included).
for _ in range(3000):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 12)))
    sam = build_sam(s)
    subs = all_substrings(s)
    assert sam.accepts(''), s                     # empty path is always valid
    for w in subs:                                # every substring is accepted
        assert sam.accepts(w), (s, w)
    for _ in range(20):                           # a sample of strings (subs and non-subs)
        w = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 6)))
        assert sam.accepts(w) == (w in subs), (s, w)
    assert len(sam.length) <= 2 * len(s)          # state count stays linear (<= 2n-1)
print('SAM accepts EXACTLY the substrings of s, states <= 2n-1: 3000 random cases OK')

SAM accepts EXACTLY the substrings of s, states <= 2n-1: 3000 random cases OK


## 3. Applications

### Count distinct substrings — `sum(len[v] - len[link[v]])`

Each state `v` (except the root) represents all substrings whose length runs from `len[link[v]] + 1` up to `len[v]` — that's exactly `len[v] - len[link[v]]` **distinct** substrings, and every distinct substring belongs to exactly one state. Summing that difference over all non-root states counts every distinct substring **in O(number of states) = O(n)** — no enumeration of the `~n²/2` substrings themselves.

### Longest common substring of two strings — run one SAM against the other

Build the SAM of `s`, then **stream `t` through it** while tracking the current matched length `l`. On each character of `t`: if the current state has that transition, follow it and `l += 1`; otherwise follow **suffix links** (shrinking `l` to `len` of the state you land on) until a transition exists or you fall to the root. The maximum `l` reached is the length of the **longest common substring (LCS)** of `s` and `t`. It's the SAM analogue of the **KMP** fall-back, and it runs in **O(|t|)** after the O(|s|) build.

In [4]:
def count_distinct_substrings(sam):
    # each non-root state contributes (len[v] - len[link[v]]) distinct substrings
    return sum(sam.length[v] - sam.length[sam.link[v]] for v in range(1, len(sam.length)))

def longest_common_substring(s, t):
    sam = build_sam(s)
    v, l, best, best_end = 0, 0, 0, 0
    for i, ch in enumerate(t):
        while v and ch not in sam.nxt[v]:
            v = sam.link[v]                       # fall back along suffix links (KMP-style)
            l = sam.length[v]                     # matched length shrinks to this state's len
        if ch in sam.nxt[v]:
            v = sam.nxt[v][ch]; l += 1
        else:
            v, l = 0, 0                           # no match even at the root
        if l > best:
            best, best_end = l, i
    return t[best_end - best + 1: best_end + 1]

print('distinct substrings of "abcbc":', count_distinct_substrings(build_sam('abcbc')))
print('distinct substrings of "banana":', count_distinct_substrings(build_sam('banana')))
print('LCS("abcbc", "dbcbf") :', repr(longest_common_substring('abcbc', 'dbcbf')))
print('LCS("banana", "ananas"):', repr(longest_common_substring('banana', 'ananas')))

distinct substrings of "abcbc": 12
distinct substrings of "banana": 15
LCS("abcbc", "dbcbf") : 'bcb'
LCS("banana", "ananas"): 'anana'


In [5]:
# PROOF: both applications match an O(n^2)/O(n^3) brute force over actual substrings.
def distinct_brute(s):
    return len(all_substrings(s))

def lcs_len_brute(s, t):
    common = all_substrings(s) & all_substrings(t)
    return max((len(w) for w in common), default=0)

for _ in range(3000):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 12)))
    assert count_distinct_substrings(build_sam(s)) == distinct_brute(s), s

for _ in range(3000):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(0, 10)))
    t = ''.join(rng.choice('abc') for _ in range(rng.randint(0, 10)))
    got = longest_common_substring(s, t)
    assert len(got) == lcs_len_brute(s, t), (s, t, got)        # right length
    assert got == '' or (got in all_substrings(s) and got in all_substrings(t)), (s, t, got)
print('distinct-count == brute, and LCS == brute (length + validity): 3000 + 3000 cases OK')

distinct-count == brute, and LCS == brute (length + validity): 3000 + 3000 cases OK


## 4. Suffix tree — the relationship, Ukkonen named, an illustration

A **suffix tree** is the **compressed trie** (radix tree) of *all suffixes* of `s` — single-child chains collapsed into one edge labelled with a whole substring (see the **tries** notebook for the trie / PATRICIA-collapse idea). Append a unique sentinel `$` so no suffix is a prefix of another and every suffix ends at its own leaf; then:

- every **leaf** is a suffix, and
- every **internal node** is a *branching* point — a substring after which the text can continue in more than one way (a repeated prefix).

**The SAM↔tree duality.** The suffix automaton and the suffix tree are mirror images: the **suffix-link tree of the SAM of `reverse(s)`** is exactly the **suffix tree of `s`** (its node structure). Both encode every substring in linear space; the SAM is a *DFA over substrings*, the suffix tree is a *trie over suffixes*. They share the headline identity below.

**The provable identity.** The number of **distinct substrings** of `s` equals the **total length of all edge labels** in the suffix tree of `s$` minus `n + 1` (the `$`-only and `$`-terminated pieces). Equivalently, `total_edge_length(suffixtree(s$)) == distinct_substrings(s$)`. We build a naive **O(n²)** suffix tree (insert each suffix into a compressed trie, splitting edges as needed) purely to *see* the structure and check that identity against the SAM's distinct-substring count. Production code uses **Ukkonen's algorithm** for a true **O(n)** online build — we name it here rather than grinding through its active-point bookkeeping.

In [6]:
class SuffixTree:
    """Naive O(n^2) compressed-trie build (insert every suffix). Illustration only."""
    def __init__(self, s):
        self.root = {}                            # first-char -> [edge_label, child_dict]
        for i in range(len(s)):
            self._insert(s[i:])

    def _insert(self, suf):
        node = self.root
        while suf:
            ch = suf[0]
            if ch not in node:
                node[ch] = [suf, {}]              # brand-new leaf edge for the rest of suf
                return
            label, child = node[ch]
            k = 0                                 # length of common prefix of label and suf
            while k < len(label) and k < len(suf) and label[k] == suf[k]:
                k += 1
            if k == len(label):
                suf, node = suf[k:], child        # consumed the whole edge -> descend
            else:                                 # split the edge at position k
                mid = {label[k]: [label[k:], child]}
                node[ch] = [label[:k], mid]
                if suf[k:]:
                    mid[suf[k]] = [suf[k:], {}]
                return

    def total_edge_length(self):
        total = 0
        stack = [self.root]
        while stack:
            for label, child in stack.pop().values():
                total += len(label)
                stack.append(child)
        return total

st = SuffixTree('banana$')
def show(node, depth=0):
    for ch in sorted(node):
        label, child = node[ch]
        print('  ' * depth + repr(label))
        show(child, depth + 1)

print('suffix tree of "banana$":')
show(st.root)

suffix tree of "banana$":
'$'
'a'
  '$'
  'na'
    '$'
    'na$'
'banana$'
'na'
  '$'
  'na$'


In [7]:
# PROOF: suffix-tree total edge length == distinct substrings, and the SAM agrees.
def distinct_brute(s):
    return len(all_substrings(s))

for _ in range(2000):
    s = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 10)))
    tree_edges = SuffixTree(s + '$').total_edge_length()
    assert tree_edges == distinct_brute(s + '$'), s            # tree identity
    # and the SAM's distinct count for s lines up: distinct(s$) = distinct(s) + (n+1)
    assert distinct_brute(s + '$') == count_distinct_substrings(build_sam(s)) + len(s) + 1, s
print('suffix-tree edge-length == distinct substrings, consistent with the SAM: 2000 cases OK')

suffix-tree edge-length == distinct substrings, consistent with the SAM: 2000 cases OK


## 5. When to use what — suffix array vs automaton vs tree

All three index *every* substring of a text in linear space; they differ in what's natural to ask and how heavy the build is.

| You need... | Reach for | Why |
|---|---|---|
| Substring search, longest-repeat, k-th substring, simple + cache-friendly | **suffix array + LCP** (suffix structures) | one sorted `int` array; binary-searchable; lightest constants |
| **Online** build, **count distinct substrings**, **LCS of two strings**, automaton-style streaming | **suffix automaton** | minimal DFA in O(n); `len - len[link]` and suffix-link fall-back make these one-liners |
| Explicit tree of suffixes: branching repeats, generalized multi-string queries, on-tree DFS | **suffix tree** | nodes *are* the repeats; strongest model, heaviest constants & code |
| Many *patterns* vs streaming text | **Aho-Corasick** (suffix structures) | preprocess the pattern set, not the text |
| Plain `pattern in text` in Python | built-in `in` / `.find` | C `fastsearch` beats any pure-Python index |

Rules of thumb:

- **Suffix array** is the default — smallest, simplest, and enough for most "query a fixed text" jobs.
- **Suffix automaton** wins when the task is naturally an *automaton walk*: distinct-substring counts, longest common substring of two (or more) strings, or appending characters online.
- **Suffix tree** is the most expressive but the most code and the heaviest constants; in practice the SAM or suffix array covers the same ground with less pain. The SAM of `reverse(s)` *is* the suffix tree's skeleton, so you rarely need to build the tree explicitly.

## 6. Complexity cheat-sheet

<sub>n = |s|, m = query length, |t| = second-string length. Σ = alphabet size (per-state `dict` cost).</sub>

| Structure | Build | Key query | Space | Notes |
|---|:---:|:---:|:---:|---|
| Suffix array + LCP (suffix structures) | O(n log²n) / O(n) SA-IS | O(m log n) substring | O(n) | lightest; sorted `int` array |
| **Suffix automaton** | **O(n) online** | O(m) accept; O(\|t\|) LCS | O(n·Σ) | <= 2n-1 states, <= 3n-4 edges |
| · count distinct substrings | — | **O(n)** total | — | `sum(len[v] - len[link[v]])` |
| · longest common substring of two | — | **O(\|t\|)** after build | — | stream t, fall back on suffix links |
| Suffix tree (Ukkonen) | O(n) online | O(m) substring | O(n·Σ) | nodes = branching repeats; heaviest |
| Suffix tree (naive, shown here) | O(n²) | O(m) | O(n²) | illustration only |

<sub>The suffix automaton's headline numbers — `<= 2n-1` states and `<= 3n-4` transitions — are why a linear machine can encode up to `~n²/2` distinct substrings. For a one-off `pattern in text`, the C `in`/`.find` of the **string algorithms** notebook still wins; reach for these structures only when one text (or one pair of texts) feeds *many* substring questions.</sub>